### Importing Libraries

In [1]:
# ─────────────────────────────────────────────────────────────
# 0.  STANDARD LIBRARY
# ─────────────────────────────────────────────────────────────
import os
import sys
import json
import random
import logging
import warnings
import subprocess
from pathlib import Path
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 1.  THIRD-PARTY — NUMERIC / DATA
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────
# 2.  COMPUTER VISION & AUDIO
# ─────────────────────────────────────────────────────────────
import cv2
import librosa
import librosa.display

# ─────────────────────────────────────────────────────────────
# 3.  MACHINE LEARNING — SKLEARN
# ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# ─────────────────────────────────────────────────────────────
# 4.  DEEP LEARNING — TENSORFLOW / KERAS
# ─────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import Model, Input, regularizers
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, GlobalAveragePooling2D,
    GaussianNoise, Activation, Concatenate, Add
)
from tensorflow.keras.applications import DenseNet121, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.constraints import MaxNorm

# ─────────────────────────────────────────────────────────────
# 5.  VISUALISATION
# ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────────────────────
# 6.  SUPPRESS NOISE & REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

### Environment Setup

In [2]:
# ==============================================================
# SECTION 1 - GLOBAL CONFIGURATION
# ==============================================================
MASTER_CLASSES = ["asthma", "copd", "pneumonia"]
MASTER_LE = LabelEncoder()
MASTER_LE.fit(MASTER_CLASSES)

NUM_CLASSES    = len(MASTER_CLASSES)
IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
EPOCHS         = 30

BASE_DIR   = Path("respiratory_global_pipeline")
DATA_DIR   = BASE_DIR / "data"
MODEL_DIR  = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

In [3]:
for d in [DATA_DIR, MODEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TABULAR_FEATURES = [
    "age", "gender_enc", "smoking_status",
    "fev1_percent", "spo2", "respiratory_rate",
    "cough_severity", "wheeze", "chest_tightness",
    "crackles", "fever", "bmi", "copd_exacerbations",
]
CLASSES_TO_DROP = ["bronchitis", "lung_cancer"]
FUSION_WEIGHTS = {"image": 0.50, "tabular": 0.35, "audio": 0.15}

print(f"  Master classes : {MASTER_CLASSES}")
print(f"  Label encoding : {dict(zip(MASTER_CLASSES, MASTER_LE.transform(MASTER_CLASSES)))}")

  Master classes : ['asthma', 'copd', 'pneumonia']
  Label encoding : {'asthma': np.int64(0), 'copd': np.int64(1), 'pneumonia': np.int64(2)}


### Importing Dataset from kaggle

In [4]:
if not os.path.exists('/root/.kaggle'):
    os.makedirs('/root/.kaggle')

!mv kaggle.json /root/.kaggle/ 2>/dev/null || true
!chmod 600 /root/.kaggle/kaggle.json 2>/dev/null || true

#### Image Dataset

In [ ]:
nih_dir = DATA_DIR / "nih_chestxray"
nih_dir.mkdir(parents=True, exist_ok=True)

if not any(nih_dir.iterdir()):
    print("[Kaggle] Downloading NIH dataset...")
    os.system(f"kaggle datasets download -d nih-chest-xrays/data -p {nih_dir} --unzip")

[Kaggle] Downloading NIH dataset...


#### Audio Dataset

In [ ]:
icbhi_dir = DATA_DIR / "icbhi"
icbhi_dir.mkdir(parents=True, exist_ok=True)

if not any(icbhi_dir.iterdir()):
    print("[Kaggle] Downloading ICBHI dataset...")
    os.system(f"kaggle datasets download -d vbookshelf/respiratory-sound-database -p {icbhi_dir} --unzip")

#### Tabular Dataset

In [ ]:
tabular_csv_path = Path('/content/respiratory_pipeline/data/clinical_data.csv')

if not tabular_csv_path.exists():
    print(f"[Warning] Tabular CSV not found at {tabular_csv_path}. Please upload it before running the pipeline.")

### Preprocessing on Image dataset

In [ ]:
NIH_LABEL_MAP = {"Emphysema": "copd", "Pneumonia": "pneumonia", "Asthma": "asthma"}

def load_and_filter_nih_metadata(image_dir: Path) -> pd.DataFrame:
    meta_path = next(image_dir.rglob("Data_Entry_2017*.csv"), None)
    if meta_path is None: return pd.DataFrame(columns=["filepath", "label"])

    df = pd.read_csv(meta_path)
    df = df[df["Finding Labels"].isin(NIH_LABEL_MAP.keys())].copy()
    df["label"] = df["Finding Labels"].map(NIH_LABEL_MAP).str.lower()

    path_lookup  = {p.name: str(p) for p in image_dir.rglob("*.png")}
    df["filepath"]  = df["Image Index"].map(path_lookup)
    return df.dropna(subset=["filepath"])[["filepath", "label"]]

In [ ]:
def preprocess_chest_xray(image_path: str, target_size: tuple = IMG_SIZE) -> np.ndarray:
    img_gray  = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None: raise ValueError(f"Cannot load: {image_path}")
    clahe     = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img_gray)
    img_rgb   = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB)
    img_res   = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_AREA)
    return (img_res / 255.0).astype(np.float32)

In [ ]:
class NIHImageGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=BATCH_SIZE, augment=False, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle: np.random.shuffle(self.indices)
        self.aug = ImageDataGenerator(
            rotation_range=10, width_shift_range=0.08,
            height_shift_range=0.08, zoom_range=0.08, horizontal_flip=True,
        )

    def __len__(self): return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size: (idx + 1) * self.batch_size]
        batch_df  = self.df.iloc[batch_idx]
        X, y = [], []
        for _, row in batch_df.iterrows():
            try: img = preprocess_chest_xray(row["filepath"])
            except Exception: img = np.zeros((*IMG_SIZE, 3), dtype=np.float32)
            X.append(img)
            y.append(row["label_enc"])
        X = np.array(X, dtype=np.float32)
        y = to_categorical(np.array(y), num_classes=NUM_CLASSES)
        if self.augment:
            X = self.aug.flow(X, batch_size=len(X), shuffle=False).next()
        return X, y

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

### Built Image Model

In [ ]:
def build_image_model(learning_rate: float = 1e-4) -> Model:
    base = DenseNet121(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    for layer in base.layers[-20:]: layer.trainable = True

    inputs  = Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x)
    outputs = Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs, name="DenseNet121_ChestXray")
    model.compile(optimizer=AdamW(learning_rate=learning_rate, weight_decay=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
def train_image_model(image_dir: Path):
    print("\n" + "-" * 60)
    print("  IMAGE BRANCH - DenseNet121")
    print("-" * 60)

    df = load_and_filter_nih_metadata(image_dir)
    train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label_enc"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label_enc"])

    train_gen = NIHImageGenerator(train_df, augment=True,  shuffle=True)
    val_gen   = NIHImageGenerator(val_df,   augment=False, shuffle=False)
    test_gen  = NIHImageGenerator(test_df,  augment=False, shuffle=False)

    model = build_image_model()
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_image_model.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    y_true = test_df["label_enc"].values
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    f1 = f1_score(y_true, y_pred, average="macro")
    model.save(str(MODEL_DIR / "final_image_model.keras"))
    return model, {"history": history.history, "test_f1": f1}

### Preprocessing on Audio

In [ ]:
ICBHI_LABEL_MAP = {"COPD": "copd", "Pneumonia": "pneumonia", "Asthma": "asthma"}

def load_icbhi_metadata(audio_dir: Path) -> pd.DataFrame:
    diag_path = next(audio_dir.rglob("patient_diagnosis.csv"), None)
    if diag_path is None: return pd.DataFrame(columns=["filepath", "label"])

    diag_df = pd.read_csv(diag_path, header=None, names=["patient_id", "diagnosis"])
    diag_df = diag_df[diag_df["diagnosis"].isin(ICBHI_LABEL_MAP.keys())].copy()
    diag_df["label"] = diag_df["diagnosis"].map(ICBHI_LABEL_MAP)

    records = []
    for wav in audio_dir.rglob("*.wav"):
        pid = int(wav.stem.split("_")[0])
        row = diag_df[diag_df["patient_id"] == pid]
        if not row.empty:
            records.append({"filepath": str(wav), "patient_id": pid, "label": row.iloc[0]["label"]})

    df = pd.DataFrame(records).drop_duplicates(subset="filepath")
    df["label_enc"] = MASTER_LE.transform(df["label"])
    return df

In [ ]:
def wav_to_melspectrogram(filepath: str, sr: int = 22050, n_mels: int = 128, fmax: int = 2000, duration: float = 5.0, target_size: tuple = IMG_SIZE) -> np.ndarray:
    try:
        y, _ = librosa.load(filepath, sr=sr, duration=duration, mono=True)
    except Exception:
        return np.zeros((*target_size, 3), dtype=np.float32)

    target_len = int(sr * duration)
    y = np.pad(y, (0, max(0, target_len - len(y))), mode="constant")[:target_len]

    mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=fmax)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_n  = ((mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8) * 255).astype(np.uint8)
    mel_r  = cv2.resize(mel_n, target_size, interpolation=cv2.INTER_LINEAR)
    return np.stack([mel_r] * 3, axis=-1).astype(np.float32) / 255.0

In [ ]:
class ICBHIAudioGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=BATCH_SIZE, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size: (idx + 1) * self.batch_size]
        batch_df  = self.df.iloc[batch_idx]
        X = np.array([wav_to_melspectrogram(r["filepath"]) for _, r in batch_df.iterrows()], dtype=np.float32)
        y = to_categorical(batch_df["label_enc"].values, num_classes=NUM_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

### Built Model on Audio data

In [ ]:
def build_audio_model(learning_rate: float = 5e-4) -> Model:
    base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    inputs  = Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(NUM_CLASSES, activation="softmax")(x)
    model = Model(inputs, outputs, name="EfficientNetB0_Audio")
    model.compile(optimizer=AdamW(learning_rate=learning_rate, weight_decay=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
def train_audio_model(audio_dir: Path):
    print("\n" + "-" * 60)
    print("  AUDIO BRANCH - EfficientNetB0 on Mel-Spectrograms")
    print("-" * 60)

    df = load_icbhi_metadata(audio_dir)
    train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label_enc"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label_enc"])

    train_gen = ICBHIAudioGenerator(train_df, shuffle=True)
    val_gen   = ICBHIAudioGenerator(val_df,   shuffle=False)
    test_gen  = ICBHIAudioGenerator(test_df,  shuffle=False)

    model = build_audio_model()
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_audio_model.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    y_true = test_df["label_enc"].values
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    f1 = f1_score(y_true, y_pred, average="macro")
    model.save(str(MODEL_DIR / "final_audio_model.keras"))
    return model, {"history": history.history, "test_f1": f1}

### Preprocessing on Tabular dataset

In [ ]:
def load_and_prepare_tabular(csv_path: Path):
    df = pd.read_csv(csv_path)
    df = df[~df["diagnosis"].isin(CLASSES_TO_DROP)].copy()
    df["diagnosis"] = df["diagnosis"].str.lower().str.strip()
    df = df[df["diagnosis"].isin(MASTER_CLASSES)].copy()

    df["gender_enc"] = (df["gender"].str.upper() == "M").astype(int)
    df["label_enc"]  = MASTER_LE.transform(df["diagnosis"])

    X = df[TABULAR_FEATURES].values.astype(np.float32)
    y = df["label_enc"].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train)

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val   = scaler.transform(X_val).astype(np.float32)
    X_test  = scaler.transform(X_test).astype(np.float32)
    return (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler

In [ ]:
def mixup_tabular(X: np.ndarray, y: np.ndarray, alpha: float = 0.3, n_synthetic: int = 1200) -> tuple:
    rng     = np.random.default_rng(SEED)
    n       = len(X)
    lambdas = rng.beta(alpha, alpha, size=n_synthetic)
    idx_a   = rng.integers(0, n, size=n_synthetic)
    idx_b   = rng.integers(0, n, size=n_synthetic)
    lam     = lambdas[:, None]
    X_mix = lam * X[idx_a] + (1 - lam) * X[idx_b]
    oh_a  = to_categorical(y[idx_a], num_classes=NUM_CLASSES)
    oh_b  = to_categorical(y[idx_b], num_classes=NUM_CLASSES)
    y_mix = np.argmax(lambdas[:, None] * oh_a + (1 - lambdas[:, None]) * oh_b, axis=1)
    return (np.vstack([X, X_mix]).astype(np.float32), np.concatenate([y, y_mix]))

In [ ]:
def residual_mlp_block(x, units: int, dropout_rate: float, l2_reg: float = 1e-4):
    reg    = regularizers.l2(l2_reg)
    cnstr  = MaxNorm(max_value=3.0)
    skip   = x
    x = Dense(units, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(units, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Add()([x, skip])
    x = Activation("gelu")(x)
    return x

### Built Model On Tabular dataset

In [ ]:
def build_tabular_dnn(n_features: int, lr_schedule) -> Model:
    reg   = regularizers.l2(1e-4)
    cnstr = MaxNorm(max_value=3.0)
    inputs = Input(shape=(n_features,), name="clinical_features")
    x = GaussianNoise(0.02)(inputs)
    x = Dense(256, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False, name="proj_256")(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = residual_mlp_block(x, units=256, dropout_rate=0.40)

    skip_128 = Dense(128, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    skip_128 = BatchNormalization()(skip_128)
    x = Dense(128, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.35)(x)
    x = Dense(128, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Add()([x, skip_128])
    x = Activation("gelu")(x)

    skip_64 = Dense(64, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    skip_64 = BatchNormalization()(skip_64)
    x = Dense(64, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.25)(x)
    x = Dense(64, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Add()([x, skip_64])
    x = Activation("gelu")(x)

    x = Dense(32, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False, name="bottleneck")(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.20)(x)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="class_probs")(x)

    model = Model(inputs, outputs, name="Residual_TabularDNN")
    model.compile(optimizer = AdamW(learning_rate=lr_schedule, weight_decay=5e-4), loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])
    return model

In [ ]:
def train_tabular_model(csv_path: Path):
    print("\n" + "-" * 60)
    print("  TABULAR BRANCH - Residual MLP DNN")
    print("-" * 60)

    (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler = load_and_prepare_tabular(csv_path)
    X_train_aug, y_train_aug = mixup_tabular(X_train, y_train, alpha=0.3, n_synthetic=1200)

    cls_weights = compute_class_weight("balanced", classes=np.unique(y_train_aug), y=y_train_aug)
    class_weight_dict = {i: float(w) for i, w in enumerate(cls_weights)}

    steps_per_epoch   = int(np.ceil(len(X_train_aug) / BATCH_SIZE))
    lr_schedule = CosineDecayRestarts(initial_learning_rate = 5e-4, first_decay_steps = steps_per_epoch * 10, t_mul = 2.0, m_mul = 0.9, alpha = 1e-6)

    model = build_tabular_dnn(n_features=len(TABULAR_FEATURES), lr_schedule=lr_schedule)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_tabular_dnn.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(X_train_aug, y_train_aug, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=class_weight_dict, callbacks=callbacks, verbose=1)

    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    f1 = f1_score(y_test, y_pred, average="macro")
    model.save(str(MODEL_DIR / "final_tabular_dnn.keras"))
    return model, f1, scaler

In [ ]:
def predict_tabular_probs(model: Model, scaler: StandardScaler, clinical_features: dict) -> np.ndarray:
    x_raw  = np.array([[clinical_features.get(f, 0.0) for f in TABULAR_FEATURES]], dtype=np.float32)
    x_norm = scaler.transform(x_raw).astype(np.float32)
    return model.predict(x_norm, verbose=0)[0]

### Multimodal Data Alignment

In [ ]:
### Multimodal Data Alignment for Global Joint Training
def align_multimodal_datasets(image_dir: Path, audio_dir: Path, tabular_csv: Path) -> pd.DataFrame:
    """Pairs independent X-Ray, Audio, and Tabular samples by class to create perfect triplets."""
    img_df = load_and_filter_nih_metadata(image_dir)
    aud_df = load_icbhi_metadata(audio_dir)

    tab_df = pd.read_csv(tabular_csv)
    tab_df = tab_df[~tab_df["diagnosis"].isin(CLASSES_TO_DROP)].copy()
    tab_df["label"] = tab_df["diagnosis"].str.lower().str.strip()
    tab_df = tab_df[tab_df["label"].isin(MASTER_CLASSES)].copy()
    tab_df["gender_enc"] = (tab_df["gender"].str.upper() == "M").astype(int)

    unified_records = []
    for cls in MASTER_CLASSES:
        imgs = img_df[img_df["label"] == cls]["filepath"].values
        auds = aud_df[aud_df["label"] == cls]["filepath"].values
        tabs = tab_df[tab_df["label"] == cls].to_dict('records')

        min_count = min(len(imgs), len(auds), len(tabs))
        np.random.shuffle(imgs)
        np.random.shuffle(auds)
        np.random.shuffle(tabs)

        for i in range(min_count):
            record = tabs[i].copy()
            record["img_path"] = imgs[i]
            record["audio_path"] = auds[i]
            unified_records.append(record)

    unified_df = pd.DataFrame(unified_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
    unified_df["label_enc"] = MASTER_LE.transform(unified_df["label"])
    print(f"\n[Alignment] Unified Global Dataset created : {len(unified_df)} perfectly matched triplets.")
    return unified_df

In [ ]:
class GlobalMultimodalGenerator(tf.keras.utils.Sequence):
    """Yields {image_input, audio_input, tabular_input} simultaneously."""
    def __init__(self, df, tabular_scaler, batch_size=BATCH_SIZE, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.scaler = tabular_scaler
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): return int(np.ceil(len(self.df) / self.batch_size))
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_df  = self.df.iloc[batch_idx]

        X_img, X_aud, X_tab, y = [], [], [], []
        for _, row in batch_df.iterrows():
            try: X_img.append(preprocess_chest_xray(row["img_path"]))
            except: X_img.append(np.zeros((*IMG_SIZE, 3), dtype=np.float32))

            X_aud.append(wav_to_melspectrogram(row["audio_path"]))
            X_tab.append(row[TABULAR_FEATURES].values.astype(np.float32))
            y.append(row["label_enc"])

        X_tab_scaled = self.scaler.transform(np.array(X_tab))
        X_dict = {
            "image_input":   np.array(X_img, dtype=np.float32),
            "audio_input":   np.array(X_aud, dtype=np.float32),
            "tabular_input": X_tab_scaled.astype(np.float32)
        }
        y_cat = to_categorical(np.array(y), num_classes=NUM_CLASSES)
        return X_dict, y_cat

### Global (Joint) Model Architecture

In [ ]:
def build_global_model(lr_schedule) -> Model:
    """
    Assembles the unified joint model using your existing independent model functions.
    Extracts the feature vector just before the final softmax layer for deep fusion.
    """
    reg = regularizers.l2(1e-4)
    cnstr = MaxNorm(max_value=3.0)

    # 1. Instantiate your exact existing models
    img_model = build_image_model()
    aud_model = build_audio_model()
    tab_model = build_tabular_dnn(n_features=len(TABULAR_FEATURES), lr_schedule=lr_schedule)

    # 2. Extract the feature vectors (Strip the final Softmax Dense layer)
    # In all 3 of your models, layers[-1] is Dense(Softmax). layers[-2] is Dropout.
    img_feat = img_model.layers[-2].output
    aud_feat = aud_model.layers[-2].output
    tab_feat = tab_model.layers[-2].output

    # 3. GLOBAL EARLY FUSION
    merged = Concatenate(name="fusion_concat")([img_feat, aud_feat, tab_feat])

    # Joint Reasoning Layers
    x = Dense(256, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(merged)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.40)(x)

    x = Dense(128, kernel_regularizer=reg, kernel_constraint=cnstr, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.30)(x)

    # Single unified prediction head
    outputs = Dense(NUM_CLASSES, activation="softmax", name="final_prediction")(x)

    # Build Global Model mapping the independent inputs to the fused output
    model = Model(
        inputs=[img_model.input, aud_model.input, tab_model.input],
        outputs=outputs,
        name="Global_Multimodal_Net"
    )

    model.compile(
        optimizer=AdamW(learning_rate=lr_schedule, weight_decay=5e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
def train_global_model(df: pd.DataFrame):
    print("\n" + "-" * 60)
    print("  TRAINING GLOBAL MULTIMODAL NETWORK")
    print("-" * 60)

    # Stratified Splits
    train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label_enc"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label_enc"])

    # Fit Tabular Scaler ONLY on training data
    scaler = StandardScaler()
    scaler.fit(train_df[TABULAR_FEATURES].values)

    train_gen = GlobalMultimodalGenerator(train_df, scaler, shuffle=True)
    val_gen   = GlobalMultimodalGenerator(val_df, scaler, shuffle=False)
    test_gen  = GlobalMultimodalGenerator(test_df, scaler, shuffle=False)

    steps_per_epoch = int(np.ceil(len(train_df) / BATCH_SIZE))
    lr_schedule = CosineDecayRestarts(
        initial_learning_rate=3e-4,
        first_decay_steps=steps_per_epoch * 10,
        t_mul=2.0, m_mul=0.9, alpha=1e-6
    )

    model = build_global_model(lr_schedule)
    model.summary(line_length=90)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_global_model.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    print("\n[Evaluation] Testing Global Model on hold-out test set...")
    y_true = test_df["label_enc"].values
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print(classification_report(y_true, y_pred, target_names=MASTER_LE.inverse_transform([0, 1, 2])))

    plot_confusion_matrix(y_true, y_pred, "Global_Model")
    plot_training_history(history.history, "Global_Model")
    model.save(str(MODEL_DIR / "final_global_model.keras"))

    return model, scaler

### Plotting and Report Tools

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title: str) -> None:
    cm     = confusion_matrix(y_true, y_pred)
    labels = MASTER_LE.inverse_transform([0, 1, 2])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f"Confusion Matrix - {title}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout()
    safe = title.replace(" ", "_").replace("(", "").replace(")", "")
    path = REPORT_DIR / f"cm_{safe}.png"
    fig.savefig(str(path), dpi=150)
    plt.close(fig)

In [ ]:
def plot_training_history(history: dict, model_name: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history["loss"], label="Train Loss", linewidth=2)
    axes[0].plot(history["val_loss"], label="Val Loss", linewidth=2, linestyle="--")
    axes[0].set_title(f"{model_name} - Loss"); axes[0].legend()
    axes[1].plot(history["accuracy"], label="Train Acc", linewidth=2)
    axes[1].plot(history["val_accuracy"], label="Val Acc", linewidth=2, linestyle="--")
    axes[1].set_title(f"{model_name} - Accuracy"); axes[1].legend()
    plt.tight_layout()
    path = REPORT_DIR / f"history_{model_name.replace(' ', '_')}.png"
    fig.savefig(str(path), dpi=150)
    plt.close(fig)

In [ ]:
def build_joint_doctor_report(probs: np.ndarray, patient_meta: dict) -> dict:
    idx = int(np.argmax(probs))
    final_class = MASTER_LE.inverse_transform([idx])[0]
    conf_pct = float(probs[idx]) * 100

    recommendations = {
        "copd":      ["Spirometry (FEV1/FVC < 0.70 confirms obstruction)", "CT chest for emphysema", "Initiate LABA/LAMA"],
        "pneumonia": ["Chest X-ray / CT for consolidation pattern", "CBC, CRP, procalcitonin", "Empiric antibiotics"],
        "asthma":    ["Peak expiratory flow monitoring", "IgE / eosinophil count", "Initiate ICS step therapy"],
    }

    return {
        "report_metadata": {
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "pipeline_type": "Joint/Early Fusion",
            "patient_info": patient_meta,
        },
        "joint_diagnosis": {
            "final_class": final_class,
            "confidence_percent": round(conf_pct, 2),
            "probabilities": {cls: round(float(p)*100, 2) for cls, p in zip(MASTER_CLASSES, probs)},
        },
        "clinical_flags": {
            "high_uncertainty": conf_pct < 65.0,
            "human_review_needed": conf_pct < 65.0,
        },
        "clinical_recommendations": recommendations.get(final_class, []),
    }

def print_joint_report(report: dict) -> None:
    jd = report["joint_diagnosis"]
    cf = report["clinical_flags"]
    print("\n" + "=" * 60)
    print(f"  DOCTOR REPORT - Patient {report['report_metadata']['patient_info'].get('patient_id', 'Unknown')}")
    print("=" * 60)
    print(f"  Final Diagnosis      : {jd['final_class'].upper()}")
    print(f"  Confidence           : {jd['confidence_percent']:.1f}%")
    print(f"  High Uncertainty     : {cf['high_uncertainty']}")
    print(f"  Human Review Needed  : {cf['human_review_needed']}")
    print("\n  Joint Softmax Probabilities:")
    for cls, prob in jd["probabilities"].items():
        bar = "#" * int(prob / 5)
        print(f"    {cls:<12} {prob:5.1f}%  {bar}")
    print("\n  Clinical Recommendations:")
    for rec in report["clinical_recommendations"]:
        print(f"    - {rec}")
    print("=" * 60)

### Global Model Training Flow & Orchestrator

In [ ]:
def train_global_model(df: pd.DataFrame):
    print("\n" + "-" * 60)
    print("  TRAINING GLOBAL MULTIMODAL NETWORK")
    print("-" * 60)

    train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["label_enc"])
    train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=SEED, stratify=train_df["label_enc"])

    scaler = StandardScaler()
    scaler.fit(train_df[TABULAR_FEATURES].values)

    train_gen = GlobalMultimodalGenerator(train_df, scaler, shuffle=True)
    val_gen   = GlobalMultimodalGenerator(val_df, scaler, shuffle=False)
    test_gen  = GlobalMultimodalGenerator(test_df, scaler, shuffle=False)

    steps_per_epoch = int(np.ceil(len(train_df) / BATCH_SIZE))
    lr_schedule = CosineDecayRestarts(
        initial_learning_rate=3e-4,
        first_decay_steps=steps_per_epoch * 10,
        t_mul=2.0, m_mul=0.9, alpha=1e-6
    )

    model = build_global_model(lr_schedule)
    model.summary(line_length=90)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(str(MODEL_DIR / "best_global_model.keras"), monitor="val_loss", save_best_only=True, verbose=1),
    ]

    history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks, verbose=1)

    print("\n[Evaluation] Testing Global Model on hold-out test set...")
    y_true = test_df["label_enc"].values
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    print(classification_report(y_true, y_pred, target_names=MASTER_LE.inverse_transform([0, 1, 2])))
    plot_confusion_matrix(y_true, y_pred, "Global_Model")
    plot_training_history(history.history, "Global_Model")
    model.save(str(MODEL_DIR / "final_global_model.keras"))

    return model, scaler

In [ ]:
def run_global_pipeline():
    print("\n" + "=" * 70)
    print("  GLOBAL MULTIMODAL PIPELINE EXECUTION STARTED")
    print("=" * 70)

    if not tabular_csv_path.exists():
        print(f"[Error] {tabular_csv_path} not found. Please verify the file path.")
        return

    print("\n[Step 1] Aligning distinct datasets into Unified Triplets...")
    unified_df = align_multimodal_datasets(nih_dir, icbhi_dir, tabular_csv_path)

    print("\n[Step 2] Compiling and Training Global Model...")
    global_model, scaler = train_global_model(unified_df)

    print("\n[Step 3] Inference Demonstration (Single Random Patient)...")
    demo_row = unified_df.sample(1).iloc[0]

    X_dict = {
        "image_input":   np.expand_dims(preprocess_chest_xray(demo_row["img_path"]), axis=0),
        "audio_input":   np.expand_dims(wav_to_melspectrogram(demo_row["audio_path"]), axis=0),
        "tabular_input": scaler.transform([demo_row[TABULAR_FEATURES].values.astype(np.float32)])
    }

    probs = global_model.predict(X_dict, verbose=0)[0]

    # We no longer need to pass audio_f1 to the joint report because the model is unified
    report = build_joint_doctor_report(probs, patient_meta={
        "patient_id": demo_row.get("patient_id", "Demo-001"),
        "age": demo_row.get("age", 0)
    })

    print_joint_report(report)
    print("\n>>> Global Pipeline Execution Completed.")

In [ ]:
# Trigger the execution
if __name__ == "__main__":
    run_global_pipeline()